# 03 — Fine-tune XLM-RoBERTa for Mizo NER (v2)

Trains on the rebuilt `bio_v2` data: 99.3% entity coverage (was 79.5%) and
correct Mizo orthography (the previous data carried mojibake in 13.8% of
sentences).

Sentence partitions are identical to the original run, so results stay directly
comparable — only the tags and the text encoding changed.

**No pandas.** The `mizen` environment has a pandas/NumPy ABI mismatch that
breaks DataFrame construction. Plain lists of dicts are used throughout.

**Run from the repository root.** Expect roughly 8 hours on an RTX 3060 12 GB.

## Cell 1: Environment and paths

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA   = ROOT / "data" / "processed" / "bio_v2"
MODELS = ROOT / "models"
RES    = ROOT / "results" / "ner"
MODELS.mkdir(exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
OUT_MODEL = MODELS / "mizo_ner_v2"

for s in ("train", "dev", "test"):
    p = DATA / f"mizo_ner_{s}.json"
    print(("  ok   " if p.exists() else "  MISS ") + str(p.relative_to(ROOT)))
    if not p.exists():
        sys.exit("Run 02b_rebuild_bio_from_offsets.ipynb first")

print(f"\nNumPy   {np.__version__}")
print(f"PyTorch {torch.__version__}   CUDA: {torch.cuda.is_available()}")
try:
    import pandas as pd
    print(f"pandas  {pd.__version__}  (not used by this notebook)")
except Exception as e:
    print(f"pandas  unavailable: {e}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    VRAM = props.total_memory / 1024**3
    print(f"GPU: {props.name}   VRAM: {VRAM:.1f} GB")
else:
    VRAM = 0
    print("WARNING: no GPU detected. Training on CPU is not practical here.")

Repo root: C:\Users\Haulai\mizo-ner
  ok   data\processed\bio_v2\mizo_ner_train.json
  ok   data\processed\bio_v2\mizo_ner_dev.json
  ok   data\processed\bio_v2\mizo_ner_test.json

NumPy   1.26.4
PyTorch 2.7.1+cu118   CUDA: True
pandas  2.3.3  (not used by this notebook)
GPU: NVIDIA GeForce RTX 3060   VRAM: 12.0 GB


## Cell 2: Load the splits

In [2]:
def load_split(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)          # list of {"id", "tokens", "tags"}

splits = {s: load_split(DATA / f"mizo_ner_{s}.json") for s in ("train", "dev", "test")}
for s, recs in splits.items():
    print(f"  {s:<6}{len(recs):>8,} sentences")

r0 = splits["train"][0]
assert isinstance(r0["tokens"], list) and isinstance(r0["tags"], list)
print(f"\nkeys   : {list(r0.keys())}")
print(f"sample : {r0['tokens'][:6]} ...")
print(f"         {r0['tags'][:6]} ...")

bad = sum(1 for recs in splits.values() for r in recs if len(r["tokens"]) != len(r["tags"]))
print(f"token/tag length mismatches: {bad}")
assert bad == 0

entity_types = ["EVENT","FAC","GPE","LANGUAGE","LAW","LOC",
                "NORP","ORG","PERSON","PRODUCT","WORK_OF_ART"]
tag_list = ["O"] + [f"{p}-{e}" for e in entity_types for p in ("B","I")]
tag2id = {t: i for i, t in enumerate(tag_list)}
id2tag = {i: t for t, i in tag2id.items()}
print(f"\nLabels: {len(tag_list)}")

seen = {t for recs in splits.values() for r in recs for t in r["tags"]}
unknown = seen - set(tag_list)
assert not unknown, f"Unexpected tags in data: {unknown}"
print("Label set covers the data.")

json.dump({"tag2id": tag2id, "id2tag": {str(k): v for k, v in id2tag.items()}},
          open(RES / "tag_mappings_v2.json", "w"))

  train  352,941 sentences
  dev     44,118 sentences
  test    44,118 sentences

keys   : ['id', 'tokens', 'tags']
sample : ['Kar', 'thum', 'chawlh', 'Liana-an', 'a', 'la.'] ...
         ['O', 'O', 'O', 'B-PERSON', 'O', 'O'] ...
token/tag length mismatches: 0

Labels: 23
Label set covers the data.


## Cell 3: Choose the sequence length

The original run fixed this at 64 subwords. Anything longer was truncated and
entities past the cut were silently lost — the same failure we just repaired in
the BIO conversion. Measure rather than assume.

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = [r["tokens"] for r in splits["train"][:20000]]
lens = np.array([len(tokenizer(t, is_split_into_words=True)["input_ids"]) for t in sample])

print(f"Subword length over {len(lens):,} training sentences")
print(f"  mean {lens.mean():.1f}   median {np.median(lens):.0f}   max {lens.max()}")
for p in (90, 95, 99, 99.9):
    print(f"  {p:>5}th percentile: {np.percentile(lens, p):.0f}")
print()
for cand in (64, 96, 128):
    lost = int((lens > cand).sum())
    print(f"  MAX_LEN={cand:<4} truncates {lost:>6,} sentences ({lost/len(lens)*100:5.2f}%)")

if (lens > 96).mean() > 0.001:
    MAX_LEN = 128
elif (lens > 64).mean() > 0.001:
    MAX_LEN = 96
else:
    MAX_LEN = 64
print(f"\nUsing MAX_LEN = {MAX_LEN}")

Subword length over 20,000 training sentences
  mean 23.5   median 21   max 77
     90th percentile: 36
     95th percentile: 42
     99th percentile: 54
   99.9th percentile: 66

  MAX_LEN=64   truncates     26 sentences ( 0.13%)
  MAX_LEN=96   truncates      0 sentences ( 0.00%)
  MAX_LEN=128  truncates      0 sentences ( 0.00%)

Using MAX_LEN = 96


## Cell 4: Dataset

Labels attach to the first subword of each word; continuation subwords and
padding are masked out of the loss with `-100`.

In [4]:
from torch.utils.data import Dataset

class MizoNERDataset(Dataset):
    def __init__(self, records, tokenizer, max_len, tag2id):
        self.recs, self.tok = records, tokenizer
        self.max_len, self.tag2id = max_len, tag2id

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        rec = self.recs[i]
        tokens, tags = rec["tokens"], rec["tags"]
        enc = self.tok(tokens, is_split_into_words=True, max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        word_ids, labels, prev = enc.word_ids(batch_index=0), [], None
        for w in word_ids:
            if w is None:
                labels.append(-100)
            elif w != prev:
                labels.append(self.tag2id[tags[w]])
            else:
                labels.append(-100)
            prev = w
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(labels)}

ds = {s: MizoNERDataset(splits[s], tokenizer, MAX_LEN, tag2id) for s in splits}

probe = ds["train"][0]
print("shapes:", {k: tuple(v.shape) for k, v in probe.items()})
n_lab = int((probe["labels"] != -100).sum())
n_words = len(splits["train"][0]["tokens"])
print(f"labelled positions: {n_lab}   words in sentence: {n_words}")
assert n_lab <= n_words, "more labels than words - alignment is wrong"
print("Dataset OK.")

shapes: {'input_ids': (96,), 'attention_mask': (96,), 'labels': (96,)}
labelled positions: 6   words in sentence: 6
Dataset OK.


## Cell 5: Batch size

The original run used per-device 32 with gradient accumulation 2 — an effective
batch of 64 — because 4 GB left no choice. With 12 GB the same effective batch
fits in one step: identical optimization, roughly twice the speed.

In [5]:
if VRAM >= 10:
    BATCH, ACCUM = 64, 1
elif VRAM >= 7:
    BATCH, ACCUM = 32, 2
else:
    BATCH, ACCUM = 16, 4

EPOCHS, LR = 5, 2e-5
steps_per_epoch = len(ds["train"]) // (BATCH * ACCUM)

print(f"per-device batch : {BATCH}")
print(f"grad accumulation: {ACCUM}")
print(f"effective batch  : {BATCH * ACCUM}   (original run: 64)")
print(f"max sequence len : {MAX_LEN}")
print(f"steps/epoch      : {steps_per_epoch:,}")
print(f"total steps      : {steps_per_epoch * EPOCHS:,}")

per-device batch : 64
grad accumulation: 1
effective batch  : 64   (original run: 64)
max sequence len : 96
steps/epoch      : 5,514
total steps      : 27,570


## Cell 6: Train

In [6]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import f1_score, precision_score, recall_score

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(tag_list), id2label=id2tag, label2id=tag2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true, pred = [], []
    for p_row, l_row in zip(preds, labels):
        t, q = [], []
        for p, l in zip(p_row, l_row):
            if l != -100:
                t.append(id2tag[int(l)]); q.append(id2tag[int(p)])
        true.append(t); pred.append(q)
    return {"precision": precision_score(true, pred),
            "recall":    recall_score(true, pred),
            "f1":        f1_score(true, pred)}

args = TrainingArguments(
    output_dir=str(MODELS / "_ner_v2_checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=500,
    dataloader_num_workers=0,      # Windows
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds["train"], eval_dataset=ds["dev"],
                  data_collator=DataCollatorForTokenClassification(tokenizer),
                  compute_metrics=compute_metrics)

print("Training. This will take several hours.\n")
t0 = time.time()
result = trainer.train()
hours = (time.time() - t0) / 3600
print(f"\nDone in {hours:.2f} h   final training loss {result.training_loss:.4f}")

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training. This will take several hours.



Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.096200,0.094200,0.803511,0.850764,0.826463
2,0.081800,0.079757,0.847110,0.858294,0.852665
3,0.070700,0.074668,0.847875,0.874308,0.860888
4,0.060300,0.072267,0.862439,0.876574,0.869449
5,0.054400,0.072359,0.864770,0.880799,0.872711



Done in 2.57 h   final training loss 0.0994


## Cell 7: Save the model and the per-epoch history

In [7]:
trainer.save_model(str(OUT_MODEL))
tokenizer.save_pretrained(str(OUT_MODEL))
print(f"Model -> {OUT_MODEL.relative_to(ROOT)}")

history = [h for h in trainer.state.log_history if "eval_f1" in h]
train_losses = {round(h["epoch"]): h["loss"]
                for h in trainer.state.log_history if "loss" in h and "eval_loss" not in h}

print(f"\n{'Epoch':<7}{'Train loss':>12}{'Dev loss':>11}{'Prec':>9}{'Recall':>9}{'F1':>9}")
rows = []
for h in history:
    ep = round(h["epoch"])
    tl = train_losses.get(ep, float("nan"))
    print(f"{ep:<7}{tl:>12.4f}{h['eval_loss']:>11.4f}"
          f"{h['eval_precision']:>9.4f}{h['eval_recall']:>9.4f}{h['eval_f1']:>9.4f}")
    rows.append({"epoch": ep, "train_loss": tl, "dev_loss": h["eval_loss"],
                 "precision": h["eval_precision"], "recall": h["eval_recall"],
                 "f1": h["eval_f1"]})

json.dump({"model": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS,
           "learning_rate": LR, "batch": BATCH, "grad_accum": ACCUM,
           "effective_batch": BATCH * ACCUM, "training_hours": round(hours, 2),
           "train_sentences": len(ds["train"]), "dev_sentences": len(ds["dev"]),
           "test_sentences": len(ds["test"]), "history": rows},
          open(RES / "training_v2.json", "w"), indent=2)
print(f"\nHistory -> {(RES / 'training_v2.json').relative_to(ROOT)}")

Model -> models\mizo_ner_v2

Epoch    Train loss   Dev loss     Prec   Recall       F1
1            0.0859     0.0942   0.8035   0.8508   0.8265
2            0.0715     0.0798   0.8471   0.8583   0.8527
3            0.0623     0.0747   0.8479   0.8743   0.8609
4            0.0559     0.0723   0.8624   0.8766   0.8694
5            0.0544     0.0724   0.8648   0.8808   0.8727

History -> results\ner\training_v2.json


## Cell 8: Quick sanity check

Not the evaluation — that is notebook 04. This only confirms the model behaves
sensibly on suffixed entities and diacritics before you invest in a full pass.

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def tag(sentence):
    toks = sentence.split()
    enc = tokenizer(toks, is_split_into_words=True, max_length=MAX_LEN,
                    truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        pred = torch.argmax(model(**enc).logits, dim=2)[0].cpu().numpy()
    wid, seen, out = enc.word_ids(batch_index=0), set(), ["O"] * len(toks)
    for pos, w in enumerate(wid):
        if w is not None and w not in seen:
            seen.add(w); out[w] = id2tag[int(pred[pos])]
    return list(zip(toks, out))

tests = [
    "Liana chuan Aizawl atangin Delhi a kal.",
    "Aizawlah an thuthmun tur ngaihtuah mek a ni.",
    "Lal Thanhawla chuan Congress Bhavan-ah thu a sawi.",
    "Mizo tawng hi Tibeto-Burman ṭawng a ni.",
]
for s in tests:
    ents = [(t, g) for t, g in tag(s) if g != "O"]
    print(f"\n{s}")
    for t, g in ents:
        print(f"    {t:<24}{g}")
    if not ents:
        print("    (nothing predicted)")


Liana chuan Aizawl atangin Delhi a kal.
    Liana                   B-PERSON
    Aizawl                  B-GPE
    Delhi                   B-GPE

Aizawlah an thuthmun tur ngaihtuah mek a ni.
    Aizawlah                B-GPE

Lal Thanhawla chuan Congress Bhavan-ah thu a sawi.
    Lal                     B-PERSON
    Thanhawla               I-PERSON
    Congress                B-ORG
    Bhavan-ah               I-ORG

Mizo tawng hi Tibeto-Burman ṭawng a ni.
    Mizo                    B-PERSON
